In [1]:
# InGame Score Prediction Model Notebook 2: Data Preprocessing & Helper Functions

In [ ]:
# Header For Imports

import sys
import os

# Determine the project root (assuming structure: score-genius/ with backend/ and notebooks/ as siblings)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now import other packages
import pandas as pd
import numpy as np
import joblib
from sqlalchemy import create_engine
import config
import traceback

# Import shared functions from the common module
from backend.models.features import NBAFeatureGenerator



PROJECT_ROOT: /Users/mattb/Desktop/Projects/score-genius
MODEL_PATH: /Users/mattb/Desktop/Projects/score-genius/backend/models/score_prediction_model.pkl


In [3]:
# Cell 1: Load Data from Previous Notebook in Pipeline

import json

def load_data_from_pipeline(input_file='data/live_game_data.pkl'):
    """
    Load live game data from previous notebook in the pipeline.
    
    Args:
        input_file: File with saved data
    Returns:
        DataFrame with live game data
    """
    try:
        if not os.path.exists(input_file):
            print(f"No pipeline data found at {input_file}")
            return pd.DataFrame()
        
        # Load the DataFrame
        df = pd.read_pickle(input_file)
        
        # Try to load metadata
        metadata_file = input_file.replace('.pkl', '_metadata.json')
        if os.path.exists(metadata_file):
            with open(metadata_file, 'r') as f:
                metadata = json.load(f)
                print(f"Loaded pipeline data from {metadata.get('timestamp', 'unknown time')}")
                print(f"Contains {metadata.get('num_games', 'unknown')} games, {metadata.get('active_games', 'unknown')} active")
        else:
            print(f"Loaded pipeline data with {len(df)} games")
        
        return df
    
    except Exception as e:
        print(f"Error loading pipeline data: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Load data from the previous notebook
pipeline_df = load_data_from_pipeline()

if not pipeline_df.empty:
    print(f"Successfully loaded {len(pipeline_df)} games from pipeline")
    # Make data available to the rest of the notebook
    df = pipeline_df
else:
    print("No pipeline data available. Using cached or sample data if available.")

Loaded pipeline data from 2025-03-19T20:02:24.138775
Contains 3 games, 3 active
Successfully loaded 3 games from pipeline


In [4]:
# Cell 2: Imports and Helper Functions

import pandas as pd
import numpy as np
import joblib
from sqlalchemy import create_engine
import config

def convert_time_to_minutes(time_str: str) -> float:
    """
    Convert a time string "MM:SS" to a float (minutes + seconds/60).
    Returns None if conversion fails.
    """
    if pd.isna(time_str) or ":" not in str(time_str):
        return None
    try:
        minutes, seconds = str(time_str).split(":")
        return float(minutes) + float(seconds) / 60.0
    except Exception as e:
        print(f"Error converting time: {e}")
        return None
    
# Import the NBAFeatureGenerator from our unified module
from models.features import NBAFeatureGenerator

# Create an instance of the feature generator with debugging enabled
feature_generator = NBAFeatureGenerator(debug=True)

def ensure_numeric_features(df, feature_columns, flag_critical=True):
    """
    Ensures all feature columns are numeric, replacing NaN/None values with appropriate defaults.
    Also flags critical missing values for review.
    
    Args:
        df (DataFrame): Input DataFrame
        feature_columns (list): List of column names to process
        flag_critical (bool): Whether to flag critical missing values
        
    Returns:
        DataFrame: DataFrame with ensured numeric features
        dict: Dictionary of flags for critical missing values (if flag_critical=True)
    """
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Default values based on column type
    default_map = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'home_score': 0, 'away_score': 0,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'score_ratio': 0.5,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0,
        'cumulative_momentum': 0
    }
    
    # Critical columns where missing values might significantly impact predictions
    critical_columns = ['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                       'away_q1', 'away_q2', 'away_q3', 'away_q4',
                       'score_ratio', 'cumulative_momentum']
    
    # For any column not in default_map, use 0 as default
    for col in feature_columns:
        if col not in default_map:
            default_map[col] = 0
    
    # Dictionary to track critical missing values
    missing_critical = {}
    
    # Process each column
    for col in feature_columns:
        if col in result_df.columns:
            # Store original NaN count for critical columns
            if flag_critical and col in critical_columns:
                nan_count = result_df[col].isna().sum()
                if nan_count > 0:
                    missing_critical[col] = nan_count
            
            # Convert to numeric, forcing errors to NaN
            result_df[col] = pd.to_numeric(result_df[col], errors='coerce')
            
            # Replace NaN values with appropriate defaults
            result_df[col] = result_df[col].fillna(default_map.get(col, 0))
        else:
            # If column doesn't exist, add it with default values
            result_df[col] = default_map.get(col, 0)
    
    # Print summary of critical missing values if requested
    if flag_critical and missing_critical:
        print("WARNING: Critical missing values detected:")
        for col, count in missing_critical.items():
            print(f"  • {col}: {count} missing values")
    
    if flag_critical:
        return result_df, missing_critical
    else:
        return result_df

In [5]:
# Cell 2B: Helper functions for data safety and compatibility

def calculate_derived_features(df, imputation_strategy='mean'):
    """
    Calculate derived features from game data with improved imputation.
    
    Args:
        df: DataFrame with raw features
        imputation_strategy: Strategy for imputation ('mean', 'median', 'zero', or 'default')
        
    Returns:
        DataFrame: DataFrame with derived features
    """
    result_df = df.copy()
    missing_data_flags = {}
    
    # Get quarters and check for missing data
    for idx, row in result_df.iterrows():
        # Identify missing quarter data
        missing_quarters = []
        for q in range(1, 5):
            if pd.isna(row.get(f'home_q{q}')) or pd.isna(row.get(f'away_q{q}')):
                missing_quarters.append(q)
        
        if missing_quarters and row.get('current_quarter', 0) >= max(missing_quarters):
            # Flag missing data for played quarters (critical issue)
            game_id = row.get('game_id', idx)
            missing_data_flags[game_id] = f"Missing Q{missing_quarters} data despite being in Q{row.get('current_quarter')}"
    
    # 1. Calculate current scores from quarters with improved imputation
    for idx, row in result_df.iterrows():
        # Home score
        if pd.isna(row.get('home_score')) or row.get('home_score', 0) == 0:
            # Impute missing quarter scores
            quarter_scores = []
            for i in range(1, 5):
                q_val = row.get(f'home_q{i}', None)
                if pd.isna(q_val) and row.get('current_quarter', 0) >= i:
                    # Need imputation for a played quarter
                    if imputation_strategy == 'mean':
                        # Use mean of other quarters
                        valid_quarters = [float(row.get(f'home_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'home_q{j}', None))]
                        q_val = sum(valid_quarters) / len(valid_quarters) if valid_quarters else 25.0
                    elif imputation_strategy == 'median':
                        valid_quarters = [float(row.get(f'home_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'home_q{j}', None))]
                        q_val = np.median(valid_quarters) if valid_quarters else 25.0
                    else:
                        # Default to zero
                        q_val = 0
                elif pd.isna(q_val):
                    q_val = 0
                
                quarter_scores.append(float(q_val or 0))
                result_df.at[idx, f'home_q{i}'] = float(q_val or 0)
            
            home_sum = sum(quarter_scores)
            result_df.at[idx, 'home_score'] = home_sum
        
        # Away score (similar logic)
        if pd.isna(row.get('away_score')) or row.get('away_score', 0) == 0:
            # Impute missing quarter scores
            quarter_scores = []
            for i in range(1, 5):
                q_val = row.get(f'away_q{i}', None)
                if pd.isna(q_val) and row.get('current_quarter', 0) >= i:
                    # Need imputation for a played quarter
                    if imputation_strategy == 'mean':
                        # Use mean of other quarters
                        valid_quarters = [float(row.get(f'away_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'away_q{j}', None))]
                        q_val = sum(valid_quarters) / len(valid_quarters) if valid_quarters else 25.0
                    elif imputation_strategy == 'median':
                        valid_quarters = [float(row.get(f'away_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'away_q{j}', None))]
                        q_val = np.median(valid_quarters) if valid_quarters else 25.0
                    else:
                        # Default to zero
                        q_val = 0
                elif pd.isna(q_val):
                    q_val = 0
                
                quarter_scores.append(float(q_val or 0))
                result_df.at[idx, f'away_q{i}'] = float(q_val or 0)
            
            away_sum = sum(quarter_scores)
            result_df.at[idx, 'away_score'] = away_sum
    
    # 2. Calculate score_ratio
    for idx, row in result_df.iterrows():
        home_score = float(row.get('home_score', 0))
        away_score = float(row.get('away_score', 0))
        total = home_score + away_score
        result_df.at[idx, 'score_ratio'] = (home_score / total) if total > 0 else 0.5
    
    # 3. Calculate score differential
    result_df['score_differential'] = result_df['home_score'] - result_df['away_score']
    
    # 4. Calculate half-time differentials
    result_df['first_half_diff'] = (
        result_df['home_q1'].fillna(0) + result_df['home_q2'].fillna(0) -
        result_df['away_q1'].fillna(0) - result_df['away_q2'].fillna(0)
    )
    
    result_df['pre_q4_diff'] = (
        result_df['home_q1'].fillna(0) + result_df['home_q2'].fillna(0) + result_df['home_q3'].fillna(0) -
        result_df['away_q1'].fillna(0) - result_df['away_q2'].fillna(0) - result_df['away_q3'].fillna(0)
    )
    
    # 5. Calculate momentum features
    for col in ['q1_to_q2_momentum','q2_to_q3_momentum','q3_to_q4_momentum','cumulative_momentum']:
        if col not in result_df.columns:
            result_df[col] = 0
    
    for idx, row in result_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Initialize with zeros to ensure these columns exist
        result_df.at[idx, 'q1_to_q2_momentum'] = 0
        result_df.at[idx, 'q2_to_q3_momentum'] = 0
        result_df.at[idx, 'q3_to_q4_momentum'] = 0
        result_df.at[idx, 'cumulative_momentum'] = 0
        
        # Calculate quarter-to-quarter momentum shifts
        if current_quarter >= 2:
            # Q1 to Q2 momentum
            home_q1 = float(row.get('home_q1') or 0)
            home_q2 = float(row.get('home_q2') or 0)
            away_q1 = float(row.get('away_q1') or 0)
            away_q2 = float(row.get('away_q2') or 0)
            
            q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
            # Cap extreme values
            q1_to_q2 = max(min(q1_to_q2, 25), -25)
            result_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2
        
        if current_quarter >= 3:
            # Q2 to Q3 momentum
            home_q2 = float(row.get('home_q2') or 0)
            home_q3 = float(row.get('home_q3') or 0)
            away_q2 = float(row.get('away_q2') or 0)
            away_q3 = float(row.get('away_q3') or 0)
            
            q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
            # Cap extreme values
            q2_to_q3 = max(min(q2_to_q3, 25), -25)
            result_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3
        
        if current_quarter >= 4:
            # Q3 to Q4 momentum
            home_q3 = float(row.get('home_q3') or 0)
            home_q4 = float(row.get('home_q4') or 0)
            away_q3 = float(row.get('away_q3') or 0)
            away_q4 = float(row.get('away_q4') or 0)
            
            q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
            # Cap extreme values
            q3_to_q4 = max(min(q3_to_q4, 25), -25)
            result_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4
        
        # Calculate cumulative momentum with weights
        weights = [0.2, 0.3, 0.5]  # Weights for Q1→Q2, Q2→Q3, Q3→Q4
        
        if current_quarter == 2:
            result_df.at[idx, 'cumulative_momentum'] = result_df.at[idx, 'q1_to_q2_momentum'] / 15.0
        elif current_quarter == 3:
            q1_to_q2 = result_df.at[idx, 'q1_to_q2_momentum']
            q2_to_q3 = result_df.at[idx, 'q2_to_q3_momentum']
            weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1]) / (weights[0] + weights[1])
            result_df.at[idx, 'cumulative_momentum'] = weighted_momentum / 15.0
        elif current_quarter >= 4:
            q1_to_q2 = result_df.at[idx, 'q1_to_q2_momentum']
            q2_to_q3 = result_df.at[idx, 'q2_to_q3_momentum']
            q3_to_q4 = result_df.at[idx, 'q3_to_q4_momentum']
            weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1] + q3_to_q4 * weights[2]) / sum(weights)
            result_df.at[idx, 'cumulative_momentum'] = weighted_momentum / 15.0
        
        # Normalize momentum to [-1, 1]
        result_df.at[idx, 'cumulative_momentum'] = max(min(result_df.at[idx, 'cumulative_momentum'], 1.0), -1.0)
    
    # 6. Add time remaining
    result_df['time_remaining_mins'] = result_df['current_quarter'].apply(
        lambda q: max(0, 48 - ((int(q) - 1) * 12 + 6))  # Approximate minutes left
    )
    result_df['time_remaining_norm'] = result_df['time_remaining_mins'] / 48.0
    
    # Print report on missing data
    if missing_data_flags:
        print("\nWARNING: Missing data detected in critical game states:")
        for game_id, message in missing_data_flags.items():
            print(f"  • Game {game_id}: {message}")
    
    return result_df

def prepare_features_for_model(df, expected_features, team_avgs=None):
    """
    Prepares features in the correct format for model input.
    
    Args:
        df (DataFrame): Input game data
        expected_features (list): Features expected by the model
        team_avgs (dict, optional): Team scoring averages for rolling features
        
    Returns:
        DataFrame: Features ready for model prediction
    """
    # Ensure all basic numeric features are present and valid
    df = ensure_numeric_features(df, expected_features)
    
    # If team_avgs provided, use them for rolling averages
    if team_avgs is not None and 'rolling_home_score' in expected_features:
        # Update rolling average features if in expected features
        for idx, row in df.iterrows():
            if 'home_team' in row and row['home_team'] in team_avgs:
                df.at[idx, 'rolling_home_score'] = team_avgs[row['home_team']]
            if 'away_team' in row and row['away_team'] in team_avgs:
                df.at[idx, 'rolling_away_score'] = team_avgs[row['away_team']]
    
    # Select only the expected features in the correct order
    X = df[expected_features].copy()
    
    return X

In [6]:
# Cell 2C: Optimized Rest Days Calculation with Statistical Validation

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine
import config
import traceback

def calculate_improved_rest_features(df: pd.DataFrame, max_lookback_days: int = 30, 
                                    data_driven_bounds: bool = True) -> pd.DataFrame:
    """
    Calculate rest-related features with better validation, memory efficiency and data-driven bounds.
    Returns the input DataFrame augmented with rest_days_home, rest_days_away,
    is_back_to_back flags, and rest_advantage.
    
    Args:
        df: Input DataFrame with game data
        max_lookback_days: Maximum days to look back for previous games
        data_driven_bounds: Use statistical analysis to determine reasonable bounds
        
    Returns:
        DataFrame with added rest features
    """
    print(f"Calculating rest features with optimized approach...")
    if df.empty:
        print("No data provided for rest calculation")
        return df
    
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Early exit if no game_date
    if 'game_date' not in result_df.columns:
        print("Warning: 'game_date' column not found. Using default rest values.")
        result_df['rest_days_home'] = 2.0
        result_df['rest_days_away'] = 2.0
        result_df['is_back_to_back_home'] = 0
        result_df['is_back_to_back_away'] = 0
        result_df['rest_advantage'] = 0
        return result_df
    
    # Ensure datetime format and sort by date
    try:
        result_df['game_date'] = pd.to_datetime(result_df['game_date'], errors='coerce')
        invalid_dates = result_df['game_date'].isna().sum()
        if invalid_dates > 0:
            print(f"Warning: {invalid_dates} invalid dates found; filling with median date.")
            median_date = result_df['game_date'].median()
            result_df['game_date'] = result_df['game_date'].fillna(median_date)
    except Exception as e:
        print(f"Error processing dates: {e}")
        traceback.print_exc()
        # Default values if date processing fails
        result_df['rest_days_home'] = 2.0
        result_df['rest_days_away'] = 2.0
        result_df['is_back_to_back_home'] = 0
        result_df['is_back_to_back_away'] = 0
        result_df['rest_advantage'] = 0
        return result_df
    
    # Initialize rest features with default values
    result_df['rest_days_home'] = 2.0
    result_df['rest_days_away'] = 2.0
    result_df['is_back_to_back_home'] = 0
    result_df['is_back_to_back_away'] = 0
    
    # Create auxiliary structures for efficient lookups
    # Sort once, then use team-specific indices
    result_df = result_df.sort_values('game_date')
    
    # Extract all unique teams
    all_teams = set()
    if 'home_team' in result_df.columns:
        all_teams.update(result_df['home_team'].dropna().unique())
    if 'away_team' in result_df.columns:
        all_teams.update(result_df['away_team'].dropna().unique())
    
    # Process each team's games to calculate rest days
    for team in all_teams:
        # Get indices of all games where this team appears
        home_mask = (result_df['home_team'] == team)
        away_mask = (result_df['away_team'] == team)
        team_games = result_df[home_mask | away_mask].sort_values('game_date')
        
        if len(team_games) <= 1:
            continue  # Not enough games to calculate rest
        
        # Calculate rest days for each game (except the first one)
        for i in range(1, len(team_games)):
            current_game = team_games.iloc[i]
            prev_game = team_games.iloc[i-1]
            
            # Calculate days between games
            days_rest = (current_game['game_date'] - prev_game['game_date']).days
            
            # Apply to the correct column based on home/away
            if current_game['home_team'] == team:
                idx = current_game.name  # Get original index in result_df
                result_df.at[idx, 'rest_days_home'] = days_rest
                result_df.at[idx, 'is_back_to_back_home'] = 1 if days_rest == 1 else 0
            elif current_game['away_team'] == team:
                idx = current_game.name  # Get original index in result_df
                result_df.at[idx, 'rest_days_away'] = days_rest
                result_df.at[idx, 'is_back_to_back_away'] = 1 if days_rest == 1 else 0
    
    # Determine suitable bounds for rest advantage
    if data_driven_bounds:
        # Calculate rest advantage directly
        result_df['rest_advantage'] = result_df['rest_days_home'] - result_df['rest_days_away']
        
        # Calculate statistically reasonable bounds (3 standard deviations)
        rest_advantage_mean = result_df['rest_advantage'].mean()
        rest_advantage_std = result_df['rest_advantage'].std()
        
        min_bound = max(-5, rest_advantage_mean - 3 * rest_advantage_std)
        max_bound = min(5, rest_advantage_mean + 3 * rest_advantage_std)
        
        # Apply data-driven bounds for rest advantage
        result_df['rest_advantage'] = result_df['rest_advantage'].clip(min_bound, max_bound)
        
        print(f"Data-driven rest advantage bounds: [{min_bound:.1f}, {max_bound:.1f}]")
    else:
        # Use traditional fixed bounds
        result_df['rest_advantage'] = (result_df['rest_days_home'] - result_df['rest_days_away']).clip(-5, 5)
    
    # Validate the calculated rest features
    max_rest = max(result_df['rest_days_home'].max(), result_df['rest_days_away'].max())
    if max_rest > 10:
        # Cap extremely long rest periods
        result_df['rest_days_home'] = result_df['rest_days_home'].clip(1, 10)
        result_df['rest_days_away'] = result_df['rest_days_away'].clip(1, 10)
        print(f"Warning: Detected unusually long rest periods (max: {max_rest} days)")
        print("Rest days have been capped at 10 for consistency")
    
    # Ensure rest days are at least 1
    result_df['rest_days_home'] = np.maximum(result_df['rest_days_home'], 1)
    result_df['rest_days_away'] = np.maximum(result_df['rest_days_away'], 1)
    
    # Calculate summary statistics
    b2b_home = result_df['is_back_to_back_home'].sum()
    b2b_away = result_df['is_back_to_back_away'].sum()
    total_games = len(result_df)
    
    print(f"Rest day calculation complete for {total_games} games")
    print(f"Back-to-back games: {b2b_home} home ({b2b_home/total_games*100:.1f}%), "
          f"{b2b_away} away ({b2b_away/total_games*100:.1f}%)")
    
    return result_df

# Function to analyze rest day distribution for data-driven bounds
def analyze_rest_advantages(df):
    """Analyze the distribution of rest advantages to inform reasonable bounds"""
    if 'rest_days_home' not in df.columns or 'rest_days_away' not in df.columns:
        print("Rest day columns not found in DataFrame")
        return
        
    # Calculate rest advantage
    df['rest_advantage'] = df['rest_days_home'] - df['rest_days_away']
    
    # Calculate statistics
    rest_adv_stats = df['rest_advantage'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    print("\nRest Advantage Statistics:")
    print(rest_adv_stats)
    
    # Suggest bounds based on percentiles (e.g., 1st to 99th)
    lower_bound = rest_adv_stats['1%']
    upper_bound = rest_adv_stats['99%']
    
    # Round to nearest integer for simplicity
    lower_bound = np.floor(lower_bound)
    upper_bound = np.ceil(upper_bound)
    
    print(f"\nSuggested rest advantage bounds: [{lower_bound}, {upper_bound}]")
    print(f"Traditional bounds: [-5, 5]")
    
    # Check if traditional bounds are reasonable
    if lower_bound < -5 or upper_bound > 5:
        print("Warning: Data suggests the traditional bounds [-5, 5] may be too restrictive")
        print(f"         Consider using [{lower_bound}, {upper_bound}] instead")
    else:
        print("The traditional bounds [-5, 5] appear reasonable for this data")
    
    return {'lower': lower_bound, 'upper': upper_bound}

In [ ]:
# Cell 2D: Integrated Basketball Analytics Metrics Pipeline

def convert_time_to_minutes(time_str):
    """
    Convert time string in MM:SS format to numeric minutes.
    
    Args:
        time_str: String in format 'MM:SS' (e.g., '24:35')
    Returns:
        Float representing total minutes (e.g., 24.583)
    """
    if not time_str or not isinstance(time_str, str):
        return 0.0
    
    try:
        parts = time_str.split(':')
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return minutes + (seconds / 60.0)
        else:
            return float(time_str)  # Try direct conversion if not in MM:SS format
    except (ValueError, TypeError):
        print(f"Warning: Could not convert time value '{time_str}' to minutes")
        return 0.0

def ensure_numeric_features(df, columns=None):
    """
    Ensure specified columns are numeric, converting them if necessary.
    
    Args:
        df: DataFrame with columns to convert
        columns: List of column names to convert (if None, tries to convert all)
    Returns:
        DataFrame with converted numeric columns
    """
    result_df = df.copy()
    
    # If no columns specified, try to convert all
    if columns is None:
        columns = result_df.columns
    
    for col in columns:
        if col in result_df.columns:
            # Try to convert to numeric, coercing errors to NaN
            result_df[col] = pd.to_numeric(result_df[col], errors='coerce')
            
            # Fill NaN values with 0
            if result_df[col].isna().any():
                print(f"Warning: Column '{col}' contains non-numeric values that were converted to 0")
                result_df[col] = result_df[col].fillna(0)
    
    return result_df

def integrated_basketball_analytics_pipeline(df):
    """
    Comprehensive pipeline for calculating basketball analytics metrics.
    Combines shooting, free throw, possessions, pace, and efficiency metrics.
    
    Args:
        df: DataFrame with raw game data including field goals, free throws
    Returns:
        DataFrame with all basketball analytics metrics added
    """
    print("Running integrated basketball analytics pipeline...")
    
    # Guard against None or empty DataFrame
    if df is None or len(df) == 0:
        print("Warning: Empty dataframe provided. Returning empty result.")
        return pd.DataFrame()
        
    result_df = df.copy()
    
    # Ensure key score columns are numeric
    score_cols = ['home_score', 'away_score']
    if all(col in result_df.columns for col in score_cols):
        result_df = ensure_numeric_features(result_df, score_cols)
    else:
        print("Warning: Score columns missing. Some calculations may fail.")
    
    # Track original column count to report additions
    original_column_count = len(result_df.columns)
    
    # 1. Calculate shooting metrics
    #result_df = calculate_shooting_metrics(result_df)
    
    # 2. Calculate free throw metrics
    #result_df = calculate_free_throw_metrics(result_df)
    
    # 3. Calculate advanced metrics
    #result_df = calculate_advanced_metrics(result_df)
    
    # 4. Calculate derived and interaction metrics
    # Momentum-efficiency interaction: Amplifies momentum impact when efficiency differential supports it
    if all(col in result_df.columns for col in ['cumulative_momentum', 'home_off_efficiency', 'away_off_efficiency']):
        # Momentum-efficiency interaction (high momentum + high efficiency is extra strong)
        result_df['momentum_efficiency'] = result_df['cumulative_momentum'] * (
            result_df['home_off_efficiency'] - result_df['away_off_efficiency']) / 100
        print("Added momentum_efficiency interaction metric")
    else:
        print("Skipping momentum_efficiency calculation (required columns missing)")
    
    # Pace-adjusted score differential: Normalizes point differential based on game pace
    if all(col in result_df.columns for col in ['score_differential', 'game_pace']):
        # Pace-adjusted score differential
        # (Same point differential has different meaning in slow vs fast-paced games)
        result_df['pace_adj_diff'] = result_df['score_differential'] * (100 / result_df['game_pace'].clip(lower=1))
        print("Added pace_adj_diff metric")
    elif 'score_differential' not in result_df.columns and all(col in result_df.columns for col in ['home_score', 'away_score']):
        # Create score differential if it doesn't exist
        result_df['score_differential'] = result_df['home_score'] - result_df['away_score']
        # Check if game_pace exists now
        if 'game_pace' in result_df.columns:
            result_df['pace_adj_diff'] = result_df['score_differential'] * (100 / result_df['game_pace'].clip(lower=1))
            print("Added score_differential and pace_adj_diff metrics")
        else:
            print("Added score_differential metric only (game_pace missing)")
    else:
        print("Skipping pace_adj_diff calculation (required columns missing)")
    
    # 5. Calculate form metrics for consistency if current_form data is available
    if 'current_form' in result_df.columns:
        result_df = calculate_form_metrics(result_df)
    else:
        print("Skipping form metrics calculation (current_form column missing)")
    
    # Report on metrics added
    metrics_added = len(result_df.columns) - original_column_count
    print(f"Basketball analytics pipeline complete.")
    print(f"Added {metrics_added} new metrics")
    
    return result_df


def calculate_form_metrics(df):
    """
    Calculate team form metrics based on current_form field (W/L pattern).
    
    Form metrics quantify a team's recent performance by assigning weighted values
    to win/loss patterns, with more recent games having greater influence.
    
    Args:
        df: DataFrame with raw game data including current_form
    Returns:
        DataFrame with form metrics added:
        - home_form_score: Weighted score (-1 to +1) of home team's recent performance
        - away_form_score: Weighted score (-1 to +1) of away team's recent performance
        - form_score_diff: Difference between home and away form scores
        - home_streak: Current streak length (+ for wins, - for losses)
        - away_streak: Current streak length (+ for wins, - for losses)
        - streak_diff: Difference between home and away streaks
        - total_momentum: Combined in-game and form momentum (when available)
    """
    result_df = df.copy()
    
    # Check if current_form is available
    if 'current_form' not in result_df.columns:
        print("Current form data not available. Skipping form metrics calculation.")
        return result_df
    
    # Create a function to convert form string to weighted score
    def form_to_score(form_str):
        if not form_str or not isinstance(form_str, str):
            return 0
        
        # Convert string to list of results (1 for win, 0 for loss)
        results = [1 if c == 'W' else 0 for c in form_str if c in ['W', 'L']]
        
        # If no valid W/L characters found, return 0
        if not results:
            return 0
        
        # Apply recency weighting (more recent games matter more)
        weights = [2.0, 1.5, 1.0, 0.7, 0.5] # Adjust weights as needed
        
        # Ensure we have the right number of weights
        weights = weights[:len(results)]
        if len(weights) < len(results):
            weights.extend([0.5] * (len(results) - len(weights)))
        
        # Calculate weighted score
        weighted_score = sum(w * r for w, r in zip(weights, results))
        max_possible = sum(weights)
        
        # Normalize to range -1 to 1 (where 1 is all wins, -1 is all losses)
        normalized_score = (weighted_score / max_possible) * 2 - 1
        return normalized_score
    
    # Calculate recent form score for home team
    result_df['home_form_score'] = result_df['current_form'].apply(form_to_score)
    
    # If we have away team form, calculate form differential
    if 'away_current_form' in result_df.columns:
        result_df['away_form_score'] = result_df['away_current_form'].apply(form_to_score)
        result_df['form_score_diff'] = result_df['home_form_score'] - result_df['away_form_score']
    
    # Calculate streak-related features
    def get_streak(form_str):
        """Extract current streak length (positive for wins, negative for losses)"""
        if not form_str or not isinstance(form_str, str):
            return 0
        
        # Filter to only include W and L characters
        filtered_form = ''.join([c for c in form_str if c in ['W', 'L']])
        if not filtered_form:
            return 0
            
        streak = 0
        current_type = filtered_form[0]  # First character (most recent game)
        
        for char in filtered_form:
            if char == current_type:
                if char == 'W':
                    streak += 1
                else:
                    streak -= 1
            else:
                break
                
        return streak
    
    # Add streak features
    result_df['home_streak'] = result_df['current_form'].apply(get_streak)
    
    if 'away_current_form' in result_df.columns:
        result_df['away_streak'] = result_df['away_current_form'].apply(get_streak)
        result_df['streak_diff'] = result_df['home_streak'] - result_df['away_streak']
    
    # Calculate total momentum if possible
    if 'cumulative_momentum' in result_df.columns and 'home_streak' in result_df.columns:
        # Ensure momentum is numeric
        result_df = ensure_numeric_features(result_df, ['cumulative_momentum', 'home_streak'])
        
        # Combine game momentum with form momentum
        form_weight = 0.3 # How much weight to give to form vs in-game momentum
        result_df['total_momentum'] = (1 - form_weight) * result_df['cumulative_momentum'] + \
                                    form_weight * (result_df['home_streak'] / 10)
        
        # Clip to [-1, 1] range
        result_df['total_momentum'] = result_df['total_momentum'].clip(-1, 1)
        print("Added total_momentum metric combining in-game and form momentum")
    
    print("Added form-related metrics: home_form_score, home_streak")
    return result_df

# More robust test for the integrated pipeline on the dataframe if it exists


if 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    print("\nApplying integrated basketball analytics pipeline...")
    enhanced_df = integrated_basketball_analytics_pipeline(df)
    print(f"Added {len(enhanced_df.columns) - len(df.columns)} new columns")
    # Update df to use the enhanced version
    df = enhanced_df


Applying integrated basketball analytics pipeline...
Running integrated basketball analytics pipeline...
Added 3-point and effective field goal percentage metrics
Added shooting metrics: home_fg_pct, away_fg_pct, fg_pct_diff
Added free throw rate metrics: home_ft_rate, away_ft_rate, ft_rate_diff
Added free throw metrics: home_ft_pct, away_ft_pct, ft_pct_diff
Added advanced metrics: possessions, pace, and offensive efficiency
Skipping momentum_efficiency calculation (required columns missing)
Added score_differential and pace_adj_diff metrics
Skipping form metrics calculation (current_form column missing)
Basketball analytics pipeline complete.
Added 26 new metrics
Added 26 new columns


In [8]:
# Cell 2E: Enhanced Data Consistency Checks with Adaptive Validation

import pandas as pd
import numpy as np
from datetime import datetime

def validate_and_clean_features(df: pd.DataFrame, expected_features: list = None, 
                               verbose: bool = True, adaptive_ranges: bool = True) -> pd.DataFrame:
    """
    Comprehensive validation and cleaning of feature data with adaptive ranges.
    Ensures required columns exist, converts data types, fills missing values,
    and clips out-of-range values based on data distribution or configurable bounds.

    Args:
        df: Input DataFrame
        expected_features: List of column names to process
        verbose: Whether to print detailed logs
        adaptive_ranges: Use data-driven ranges for validation instead of hardcoded values
        
    Returns:
        DataFrame: DataFrame with validated and cleaned features
    """
    if df.empty:
        if verbose:
            print("No data provided for validation")
        return pd.DataFrame()
    
    clean_df = df.copy()
    
    # If no expected features are specified, choose a default set
    if expected_features is None:
        # If momentum columns exist, we assume the "enhanced" feature set
        if 'q1_to_q2_momentum' in clean_df.columns:
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
        else:
            # Otherwise, we assume the "original" feature set
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
    
    if verbose:
        print(f"Validating {len(clean_df)} rows against {len(expected_features)} expected features")
    
    # Ensure certain columns exist (game_id, home_team, away_team)
    required_game_cols = ['game_id', 'home_team', 'away_team']
    for col in required_game_cols:
        if col not in clean_df.columns:
            if verbose:
                print(f"Warning: Missing required column: {col}. Adding default values.")
            if col == 'game_id':
                clean_df[col] = np.arange(1000, 1000 + len(clean_df))
            else:
                clean_df[col] = "Unknown"
    
    # Default values for missing columns
    default_values = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'score_ratio': 0.5,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0, 'cumulative_momentum': 0,
        'current_quarter': 0,
        'home_score': 0, 'away_score': 0
    }
    
    # Ensure each expected feature exists and is numeric (except ID/team columns)
    for feature in expected_features:
        if feature not in clean_df.columns:
            if verbose:
                print(f"Adding missing feature: {feature} with default {default_values.get(feature, 0)}")
            clean_df[feature] = default_values.get(feature, 0)
        if feature not in ['game_id', 'home_team', 'away_team']:
            clean_df[feature] = pd.to_numeric(clean_df[feature], errors='coerce').fillna(default_values.get(feature, 0))
    
    # Determine reasonable validation ranges based on data if adaptive_ranges=True
    if adaptive_ranges:
        # Calculate data-driven validation ranges (mean ± 3*std, or percentile-based)
        validation_ranges = {}
        
        # Quarter scores (use 99th percentile as upper bound)
        for q in range(1, 5):
            home_q_col = f'home_q{q}'
            away_q_col = f'away_q{q}'
            
            # Combine home and away quarter scores for better statistics
            all_q_scores = pd.concat([
                clean_df[home_q_col][clean_df[home_q_col] > 0],
                clean_df[away_q_col][clean_df[away_q_col] > 0]
            ])
            
            if len(all_q_scores) > 10:  # Only if we have sufficient data
                q_min = 0  # Quarters can't score below 0
                q_max = all_q_scores.quantile(0.99)
                # Round up to nearest 5 for readability
                q_max = np.ceil(q_max / 5) * 5
                validation_ranges[f'q{q}_scores'] = (q_min, q_max)
            else:
                # Default if insufficient data
                validation_ranges[f'q{q}_scores'] = (0, 50)
                
        # Other numeric features
        for feature in expected_features:
            if feature not in ['game_id', 'home_team', 'away_team'] and feature not in validation_ranges:
                # Skip quarter scores (already handled)
                if any(q_name in feature for q_name in ['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                                                       'away_q1', 'away_q2', 'away_q3', 'away_q4']):
                    continue
                    
                # Get non-zero/non-null values for better statistics
                feature_values = clean_df[feature]
                feature_values = feature_values[feature_values != 0]
                feature_values = feature_values.dropna()
                
                if len(feature_values) > 10:  # Only if we have sufficient data
                    if feature == 'score_ratio':
                        # Score ratio is always between 0 and 1
                        validation_ranges[feature] = (0, 1)
                    elif 'momentum' in feature:
                        # Momentum features typically range from -1 to 1
                        validation_ranges[feature] = (-1, 1)
                    elif feature in ['rolling_home_score', 'rolling_away_score']:
                        # Team scores typically between 80-140
                        score_min = max(60, feature_values.quantile(0.01))
                        score_max = min(140, feature_values.quantile(0.99))
                        validation_ranges[feature] = (score_min, score_max)
                    elif feature == 'prev_matchup_diff':
                        # Matchup differential typically within ±30
                        diff_abs_max = min(50, feature_values.abs().quantile(0.99))
                        validation_ranges[feature] = (-diff_abs_max, diff_abs_max)
                    elif feature in ['rest_days_home', 'rest_days_away']:
                        # Rest days typically 1-10
                        validation_ranges[feature] = (1, 10)
                    elif feature == 'rest_advantage':
                        # Rest advantage typically within ±5
                        adv_abs_max = min(5, feature_values.abs().quantile(0.99))
                        validation_ranges[feature] = (-adv_abs_max, adv_abs_max)
                    else:
                        # Default approach for other features: use percentiles
                        feature_min = feature_values.quantile(0.01)
                        feature_max = feature_values.quantile(0.99)
                        validation_ranges[feature] = (feature_min, feature_max)
        
        if verbose:
            print("\nData-driven validation ranges:")
            for feature, (min_val, max_val) in sorted(validation_ranges.items()):
                print(f"  {feature}: [{min_val:.2f}, {max_val:.2f}]")
    else:
        # Use fixed validation ranges
        validation_ranges = {
            'q1_scores': (0, 50),
            'q2_scores': (0, 50),
            'q3_scores': (0, 50),
            'q4_scores': (0, 50),
            'score_ratio': (0, 1),
            'rolling_home_score': (80, 130),
            'rolling_away_score': (80, 130),
            'prev_matchup_diff': (-50, 50),
            'rest_days_home': (1, 10),
            'rest_days_away': (1, 10),
            'rest_advantage': (-5, 5),
            'q1_to_q2_momentum': (-20, 20),
            'q2_to_q3_momentum': (-20, 20),
            'q3_to_q4_momentum': (-20, 20),
            'cumulative_momentum': (-1, 1)
        }
    
    # Apply validation ranges
    features_clipped = 0
    for feature, feature_range in validation_ranges.items():
        min_val, max_val = feature_range
        
        # Apply to specific quarter score columns
        if feature.startswith('q') and '_scores' in feature:
            q_num = feature[1]
            home_col = f'home_q{q_num}'
            away_col = f'away_q{q_num}'
            
            for col in [home_col, away_col]:
                if col in clean_df.columns:
                    # Count out-of-range values
                    invalid_count = ((clean_df[col] < min_val) | (clean_df[col] > max_val)).sum()
                    if invalid_count > 0:
                        features_clipped += invalid_count
                        if verbose:
                            print(f"Clipping {invalid_count} invalid values in {col} to range [{min_val:.1f}, {max_val:.1f}]")
                        clean_df[col] = clean_df[col].clip(min_val, max_val)
        
        # Apply to other columns directly
        elif feature in clean_df.columns:
            invalid_count = ((clean_df[feature] < min_val) | (clean_df[feature] > max_val)).sum()
            if invalid_count > 0:
                features_clipped += invalid_count
                if verbose:
                    print(f"Clipping {invalid_count} invalid values in {feature} to range [{min_val:.1f}, {max_val:.1f}]")
                clean_df[feature] = clean_df[feature].clip(min_val, max_val)
    
    if features_clipped > 0:
        print(f"Total features clipped to valid ranges: {features_clipped}")
    
    # Calculate home_score and away_score from quarter sums if needed
    if all(f'home_q{i}' in clean_df.columns for i in range(1, 5)):
        home_sum = clean_df[[f'home_q{i}' for i in range(1, 5)]].sum(axis=1)
        if 'home_score' in clean_df.columns:
            # Only update if current value is 0 or NA
            mask = (clean_df['home_score'].isna()) | (clean_df['home_score'] == 0)
            clean_df.loc[mask, 'home_score'] = home_sum[mask]
        else:
            clean_df['home_score'] = home_sum
            
    if all(f'away_q{i}' in clean_df.columns for i in range(1, 5)):
        away_sum = clean_df[[f'away_q{i}' for i in range(1, 5)]].sum(axis=1)
        if 'away_score' in clean_df.columns:
            # Only update if current value is 0 or NA
            mask = (clean_df['away_score'].isna()) | (clean_df['away_score'] == 0)
            clean_df.loc[mask, 'away_score'] = away_sum[mask]
        else:
            clean_df['away_score'] = away_sum
    
    # Enhanced quarter inference - more robust to edge cases
    if 'current_quarter' not in clean_df.columns:
        clean_df['current_quarter'] = 0
        
    for idx, row in clean_df.iterrows():
        # Get current quarter value
        current_q = row.get('current_quarter', 0)
        
        # If current quarter is already set and valid, skip inference
        if current_q >= 1 and current_q <= 4:
            continue
            
        # Try to infer quarter from various indicators
        q_val = 0
        
        # Method 1: Check quarter scores
        for q in range(1, 5):
            home_q_val = row.get(f'home_q{q}', 0) or 0
            away_q_val = row.get(f'away_q{q}', 0) or 0
            
            # Check for any scoring activity in this quarter
            if home_q_val > 0 or away_q_val > 0:
                q_val = max(q_val, q)
                
        # Method 2: Check status field if available
        status = str(row.get('game_status', '')).lower()
        if status in ['finished', 'complete', 'final']:
            q_val = max(q_val, 4)  # Finished games are in Q4
            
        # Method 3: Check quarter-specific momentum features if available
        if row.get('q3_to_q4_momentum', 0) != 0:
            q_val = max(q_val, 4)
        elif row.get('q2_to_q3_momentum', 0) != 0:
            q_val = max(q_val, 3)
        elif row.get('q1_to_q2_momentum', 0) != 0:
            q_val = max(q_val, 2)
            
        # Method 4: Check home/away scores
        home_score = row.get('home_score', 0) or 0
        away_score = row.get('away_score', 0) or 0
        total_score = home_score + away_score
        
        # Estimate quarter based on total score
        if total_score > 0:
            if total_score >= 160:
                q_val = max(q_val, 4)  # Likely late game
            elif total_score >= 120:
                q_val = max(q_val, 3)  # Likely mid-late game
            elif total_score >= 60:
                q_val = max(q_val, 2)  # Likely early-mid game
            else:
                q_val = max(q_val, 1)  # Likely early game
                
        clean_df.at[idx, 'current_quarter'] = q_val
    
    if verbose:
        print("\nValidation summary:")
        print(f"- Processed {len(clean_df)} rows")
        
        # Count rows by inferred quarter
        quarter_counts = clean_df['current_quarter'].value_counts().sort_index()
        print("\nRows by current quarter:")
        for quarter, count in quarter_counts.items():
            print(f"  Quarter {quarter}: {count} rows")
        
        # Calculate feature statistics
        feature_stats = pd.DataFrame({
            'min': clean_df[expected_features].min(),
            'max': clean_df[expected_features].max(),
            'mean': clean_df[expected_features].mean(),
            'missing_pct': clean_df[expected_features].isna().mean() * 100
        })
        
        print("\nFeature statistics after cleaning:")
        display(feature_stats)
    
    return clean_df